# Constitution du dataset — Classification par graveur

**Projet** : Gallica Images — Illustrations des *Métamorphoses* d'Ovide  
**Date**   : Avril 2026

Ce notebook construit le dataset pour entraîner un classifieur par graveur : pour
chaque édition, récupère les illustrations (API BnF, BSB Munich IIIF, PDF local ou
Biblioteca Digital Ovidiana), les segmente avec YOLO, et les range par graveur dans
`data/editions_ovide/segmentees/`.

## Graveurs et éditions sources

| Graveur | Dossier(s) segmenté(s) | Source | Statut |
|---|---|---|---|
| Salomon, Bernard | `bois_salomon_rouille_lyon1557` | API BnF | ok |
| Solis, Virgil | `bois_solis_feyerabend_francfort1581` | BSB Munich (IIIF) | ok |
| Wickram, Jörg | `bois_wickram_behem_mayence1545` | BSB Munich (IIIF) | ok |
| Savery, Salomon | `cuivre_savery_farnaby_paris1637` | BSB Munich (IIIF) | ok |
| Tempesta, Antonio | `cuivre_tempesta_dejode_anvers1606` + `cuivre_tempesta_jansonius_amsterdam1610` | API BnF + BSB Munich | ok |
| Borcht, Pieter van der | `cuivre_borcht_plantin_anvers1591` | BSB Munich (IIIF) | ok |
| Bouche | `cuivre_bouche_blaeu_amsterdam1702` | PDF local | ok |
| Mathieu | `cuivre_mathieu_langelier_paris1619` | API BnF | ok |
| Monconet, Balthasar | `cuivre_monconet_sommaville_paris1660` | PDF local | ok |
| Eskrich, Pierre | `bois_eskrich_rouille_lyon1556` | API BnF | ok |
| Leroy II, Guillaume | `bois_leroy_gueynard_lyon1510` | BSB Munich (IIIF) | ok |
| Passe, Crispin de | `cuivre_depasse_depasse_koln1602` + `cuivre_depasse_jansonius_arnhem1607` | PDF local (Gallica) | ok |
| Monogrammiste H.T. | `cuivre_ht_molin_lyon1697` | Biblioteca Digital Ovidiana | ok — tomes 4/5/6 (claves 52/53/54) regroupés dans un seul dossier |
| Franco | `cuivre_franco_giunta_venise1584` | Biblioteca Digital Ovidiana | ok |
| Gaultier, Léonard | `cuivre_gaultier_guillemot_paris1610` + `cuivre_gaultier_veuveguillemot_paris1614` | BSB Munich (IIIF) + PDF local | ok |
| Isaac, Jaspar | `cuivre_isaac_langelier_paris1617` | PDF local | ok |
| Briot, Isaac | `cuivre_briot_drobet_lyon1628` | PDF local | ok |
| Baur, Johann Wilhelm | `cuivre_baur_sn_augsbourg1709` + `cuivre_baur_sn_vienne1639` | BSB Munich (IIIF) | ok |
| Altzenbach, Gerhardt | — | — | à numériser (exemplaire physique Bnu Strasbourg) |
| Goltzius, Hendrick | `cuivre_goltzius_goltzius_haarlem1589` | PDF local | ok — converti en niveaux de gris |
| Mulder, Joseph | — | — | à sourcer |
| Weyen, Laurent | `cuivre_weyen_barbin_paris1669` | non documentée | ok — source à repréciser |
| Blanchin, Jean | `cuivre_blanchin_berthelin_rouen1651` | non documentée | ok |
| Philippe, Pierre | `cuivre_philippe_hackiana_leyde1670` | non documentée | ok |

Le compte à jour par graveur est calculé automatiquement en section 6.

---

**Sortie :** `data/editions_ovide/segmentees/{source}/` — illustrations segmentées, une
sous-dossier par édition.

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, "..")  # remonte à notebooks/
from gallica_utils import charger_yolo, segmenter_corpus, liberer_yolo, telecharger_pages_iiif, BASE_URL

import os, requests, fitz
from io import BytesIO
from bs4 import BeautifulSoup
import torch
from PIL import Image

RACINE          = os.path.abspath("../..")   # working_dir
DOSSIER_SEG     = os.path.join(RACINE, "data", "editions_ovide", "segmentees")
DOSSIER_SOURCES = os.path.join(RACINE, "data", "editions_ovide", "sources")
DOSSIER_PDFS    = os.path.join(DOSSIER_SOURCES, "cuivre_pdfs_bruts")
YOLOV5_REPO     = os.path.join(RACINE, "yolov5_repo")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Racine  : {RACINE}")
print(f"Device  : {device}")

Racine  : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir
Device  : cuda


## 2. Fonctions de récupération par type de source

Une fonction par type de source (API BnF, BSB Munich IIIF, PDF local, Biblioteca
Digital Ovidiana) au lieu de dupliquer la même logique de téléchargement +
segmentation dans chaque cellule d'édition. Chaque cellule d'édition en section 4
se résume alors à un ou deux appels avec les identifiants qui changent.

In [6]:
def telecharger_api_bnf(ark, nom_dossier, prefixe=None):
    """
    Récupère les illustrations d'un ouvrage via l'API BnF (embeddings CLIP valides),
    les télécharge puis les segmente avec YOLO.
    Retourne None si l'API ne renvoie pas de liste d'illustrations (ouvrage introuvable,
    erreur, etc.) — l'édition est alors à considérer comme non récupérable par cette voie.
    """
    prefixe = prefixe or nom_dossier
    r = requests.get(f"{BASE_URL}/api/ouvrages/{ark}/illustrations", timeout=30)
    illustrations = r.json()
    if not isinstance(illustrations, list):
        print(f"✗ {nom_dossier} — API : {illustrations.get('message', illustrations) if isinstance(illustrations, dict) else illustrations}")
        return None

    valides = [i for i in illustrations
               if i.get("metas", {}).get("content_embedding")
               and len(i["metas"]["content_embedding"]) == 768]
    print(f"{nom_dossier} — {len(valides)} illustrations avec embedding")

    dossier_brut = os.path.join(DOSSIER_SOURCES, nom_dossier)
    os.makedirs(dossier_brut, exist_ok=True)
    pages = []
    for i, illus in enumerate(valides):
        print(f"  {i+1}/{len(valides)}...", end="\r")
        url    = illus["metas"].get("link", "")
        view   = illus.get("view_number", i)
        chemin = os.path.join(dossier_brut, f"{prefixe}_f{view:03d}.jpg")
        if os.path.exists(chemin):
            pages.append(chemin)
            continue
        try:
            img = Image.open(BytesIO(requests.get(url, timeout=15).content)).convert("RGB")
            img.save(chemin)
            pages.append(chemin)
        except Exception as e:
            print(f"\n  Erreur {view} : {e}")

    print(f"\n✓ {len(pages)} pages téléchargées")
    return segmenter_corpus(pages, os.path.join(DOSSIER_SEG, nom_dossier), modele_yolo, conf_thres=0.25)


def telecharger_bsb(bsb_id, nom_dossier, prefixe=None):
    """
    Télécharge les pages d'une édition BSB Munich (IIIF) puis les segmente avec YOLO.
    Réutilise les pages déjà présentes dans data/sources/{nom_dossier}/ si le dossier existe.
    """
    prefixe = prefixe or nom_dossier
    dossier_source = os.path.join(DOSSIER_SOURCES, nom_dossier)
    if os.path.exists(dossier_source) and len(os.listdir(dossier_source)) > 0:
        pages = sorted(
            os.path.join(dossier_source, f) for f in os.listdir(dossier_source)
            if f.endswith(".jpg")
        )
        print(f"✓ {nom_dossier} — {len(pages)} pages déjà disponibles")
    else:
        pages = telecharger_pages_iiif(
            f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest",
            dossier_source,
            prefixe=prefixe
        )
    return segmenter_corpus(pages, os.path.join(DOSSIER_SEG, nom_dossier), modele_yolo, conf_thres=0.25)


def extraire_pages_pdf(chemin_pdf, dossier_sortie, dpi=150):
    """Convertit chaque page d'un PDF (data/editions_ovide/sources/cuivre_pdfs_bruts/) en JPG."""
    os.makedirs(dossier_sortie, exist_ok=True)
    doc   = fitz.open(chemin_pdf)
    pages = []
    for i, page in enumerate(doc):
        mat    = fitz.Matrix(dpi / 72, dpi / 72)
        pix    = page.get_pixmap(matrix=mat)
        chemin = f"{dossier_sortie}/page{i+1:03d}.jpg"
        pix.save(chemin)
        pages.append(chemin)
        print(f"  {i+1}/{len(doc)}...", end="\r")
    print(f"\n✓ {len(pages)} pages extraites depuis {os.path.basename(chemin_pdf)}")
    return pages


def traiter_pdf_local(nom_fichier_pdf, nom_dossier, dpi=150):
    """Extrait un PDF local (data/editions_ovide/sources/cuivre_pdfs_bruts/) puis le segmente avec YOLO."""
    chemin_pdf = os.path.join(DOSSIER_PDFS, nom_fichier_pdf)
    if not os.path.exists(chemin_pdf):
        print(f"⚠️  {nom_fichier_pdf} introuvable dans {DOSSIER_PDFS}")
        return None
    print(f"\nTraitement : {nom_fichier_pdf}")
    pages = extraire_pages_pdf(chemin_pdf, os.path.join(DOSSIER_SOURCES, nom_dossier))
    return segmenter_corpus(pages, os.path.join(DOSSIER_SEG, nom_dossier), modele_yolo, conf_thres=0.25)


def telecharger_depuis_clave(clave, nom_dossier, base_url="http://www.ovidiuspictus.es"):
    """
    Télécharge toutes les illustrations d'un exemplaire depuis la Biblioteca Digital
    Ovidiana. Images déjà détourées — pas de segmentation YOLO nécessaire.
    """
    url  = f"{base_url}/en/ilustracionesejemplar.php?clave={clave}"
    r    = requests.get(url, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")
    imgs = [img.get("src", "") for img in soup.find_all("img")
            if "/images/images/" in img.get("src", "")]
    dossier = os.path.join(DOSSIER_SEG, nom_dossier)
    os.makedirs(dossier, exist_ok=True)
    print(f"\n{nom_dossier} — clave={clave} — {len(imgs)} illustrations")
    for i, src in enumerate(imgs):
        print(f"  {i+1}/{len(imgs)}...", end="\r")
        url_img     = f"{base_url}/{src.replace('../', '')}"
        nom_fichier = f"ex{clave}_{os.path.basename(src)}"
        chemin      = os.path.join(dossier, nom_fichier)
        if os.path.exists(chemin):
            continue
        try:
            img = Image.open(BytesIO(requests.get(url_img, timeout=15).content)).convert("RGB")
            img.save(chemin)
        except Exception as e:
            print(f"\n  Erreur {nom_fichier} : {e}")
    n = len([f for f in os.listdir(dossier) if f.endswith(".jpg")])
    print(f"\n✓ {n} illustrations sauvegardées → {nom_dossier}")
    return n


def convertir_niveaux_de_gris(nom_dossier):
    """
    Convertit un dossier segmenté en niveaux de gris. L'original couleur est
    conservé à côté, renommé en {nom_dossier}_couleur.
    """
    dossier_gris    = os.path.join(DOSSIER_SEG, nom_dossier)
    dossier_couleur = os.path.join(DOSSIER_SEG, f"{nom_dossier}_couleur")
    if os.path.exists(dossier_gris) and not os.path.exists(dossier_couleur):
        os.rename(dossier_gris, dossier_couleur)
        print(f"✓ Renommé en : {nom_dossier}_couleur")
    os.makedirs(dossier_gris, exist_ok=True)
    images = [f for f in os.listdir(dossier_couleur) if f.endswith(".jpg")]
    for nom in images:
        img = Image.open(os.path.join(dossier_couleur, nom)).convert("L").convert("RGB")
        img.save(os.path.join(dossier_gris, nom))
        print(f"  {nom}", end="\r")
    print(f"\n✓ {len(images)} illustrations converties en niveaux de gris → {nom_dossier}")

## 3. Mapping graveur → dossiers segmentés

Correspondance entre le nom du graveur (= la classe pour l'entraînement) et les
dossiers de `data/editions_ovide/segmentees/` qui lui appartiennent.

In [4]:
# Clé   : nom du graveur (sera le nom de la classe)
# Valeur : liste de dossiers dans data/editions_ovide/segmentees/

GRAVEURS = {
    "salomon"    : ["bois_salomon_rouille_lyon1557"],
    "solis"      : ["bois_solis_feyerabend_francfort1581"],
    "wickram"    : ["bois_wickram_behem_mayence1545"],
    "savery"     : ["cuivre_savery_farnaby_paris1637"],
    "de_passe"   : ["cuivre_depasse_depasse_koln1602",
                    "cuivre_depasse_jansonius_arnhem1607"],
    "eskrich"    : ["bois_eskrich_rouille_lyon1556"],
    "leroy"      : ["bois_leroy_gueynard_lyon1510"],
    "gaultier"   : ["cuivre_gaultier_guillemot_paris1610",
                    "cuivre_gaultier_veuveguillemot_paris1614"],
    "isaac"      : ["cuivre_isaac_langelier_paris1617"],
    "briot"      : ["cuivre_briot_drobet_lyon1628"],
    "goltzius"   : ["cuivre_goltzius_goltzius_haarlem1589"],
    "blanchin"   : ["cuivre_blanchin_berthelin_rouen1651"],
    "weyen"      : ["cuivre_weyen_barbin_paris1669"],
    "baur"       : ["cuivre_baur_sn_augsbourg1709",
                    "cuivre_baur_sn_vienne1639"],
    "philippe"   : ["cuivre_philippe_hackiana_leyde1670"],
    "tempesta"   : ["cuivre_tempesta_dejode_anvers1606",
                    "cuivre_tempesta_jansonius_amsterdam1610"],
    "borcht"     : ["cuivre_borcht_plantin_anvers1591"],
    "bouche"     : ["cuivre_bouche_blaeu_amsterdam1702"],
    "mathieu"    : ["cuivre_mathieu_langelier_paris1619"],
    "monconet"   : ["cuivre_monconet_sommaville_paris1660"],
    "ht"         : ["cuivre_ht_molin_lyon1697"],
    "franco"     : ["cuivre_franco_giunta_venise1584"],

    # À ajouter quand disponibles
    # "altzenbach" : Cöllen 1681 — Bnu Strasbourg, exemplaire physique à numériser
    # "mulder"     : Amsterdam 1683 — à sourcer (ressemble à philippe)
}

# Vérification rapide — dossiers manquants
for graveur, dossiers in GRAVEURS.items():
    for d in dossiers:
        if not os.path.isdir(os.path.join(DOSSIER_SEG, d)):
            print(f"⚠️  {graveur} — {d} introuvable")
print("OK!")

OK!


## 4. Téléchargement et segmentation — nouvelles sources


Une sous-section par édition. Statut ok = illustrations déjà segmentées (relancer la
cellule ne fait que compléter les manquantes). Une sous-section sans cellule de code
signifie l'un des trois cas suivants (précisé à chaque fois) :
- le lien/ARK/BSB ID n'a jamais été retrouvé (à sourcer) ;
- l'API interrogée n'a pas renvoyé de résultat exploitable (erreur, 403, ouvrage inconnu) ;
- l'édition a été récupérée par une autre voie que celle documentée ici.

In [ ]:
# Charger YOLO — uniquement si on a de nouvelles sources à segmenter
modele_yolo = charger_yolo(yolov5_repo=YOLOV5_REPO)

### Préparation des PDFs locaux (one-off, déjà exécuté)

Renommage des PDFs téléchargés depuis Gallica (noms de fichiers Gallica bruts) vers
la convention `{type}_{graveur}_{editeur}_{ville}{année}.pdf`. Sans effet si les
fichiers ont déjà été renommés ou ne sont plus dans `data/editions_ovide/sources/cuivre_pdfs_bruts/`.

In [ ]:
RENOMMAGES_PDFS = {
    "Metamorphoseon_Ovidianarum_typi_aliquot_artificiosissimè_[...]Ovide_(0043_bpt6k15218623.pdf" : "cuivre_depasse_depasse_koln1602.pdf",
    "P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wilhelm_bpt6k1522448r.pdf"                   : "cuivre_depasse_jansonius_arnhem1607.pdf",
    "[Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide_(0043_bpt6k6277348n.pdf"                 : "cuivre_gaultier_veuveguillemot_paris1614.pdf",
    "Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide_(0043_bpt6k722055.pdf"                   : "cuivre_isaac_langelier_paris1617.pdf",
}

for ancien, nouveau in RENOMMAGES_PDFS.items():
    src = os.path.join(DOSSIER_PDFS, ancien)
    dst = os.path.join(DOSSIER_PDFS, nouveau)
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"✓ {ancien[:50]}... → {nouveau}")
    else:
        print(f"  [SKIP] {ancien[:50]}... — introuvable")

### Tempesta, Antonio
##### <u>Éditions :</u> de Jode, Anvers 1606 (`btv1b54000051z`, API BnF) · Jansonius, Amsterdam 1610 (`bsb00008186`, BSB Munich)

In [ ]:
telecharger_api_bnf("btv1b54000051z", "cuivre_tempesta_dejode_anvers1606", prefixe="tempesta")
telecharger_bsb("bsb00008186", "cuivre_tempesta_jansonius_amsterdam1610", prefixe="tempesta")

### Borcht, Pieter van der
##### <u>Édition :</u> Plantin, Anvers 1591 (`bsb00004340`, BSB Munich)

In [ ]:
telecharger_bsb("bsb00004340", "cuivre_borcht_plantin_anvers1591", prefixe="borcht")

### Bouche
##### <u>Édition :</u> Blaeu, Amsterdam 1702 (PDF local)

In [ ]:
traiter_pdf_local("cuivre_bouche_blaeu_amsterdam1702.pdf", "cuivre_bouche_blaeu_amsterdam1702")

### Mathieu
##### <u>Éditions :</u> L'Angelier, Paris 1619 (`btv1b22000826`, API BnF) · Billaine, Paris 1637 (`bpt6k87019014`) — **échec** : ouvrage introuvable par l'API BnF ("not found") et HTTP 403 sur l'IIIF Gallica, abandonnée.

In [ ]:
telecharger_api_bnf("btv1b22000826", "cuivre_mathieu_langelier_paris1619", prefixe="mathieu")

### Monconet, Balthasar
##### <u>Édition :</u> Sommaville, Paris 1660 (PDF local)

In [ ]:
traiter_pdf_local("cuivre_monconet_sommaville_paris1660.pdf", "cuivre_monconet_sommaville_paris1660")

### Eskrich, Pierre
##### <u>Éditions :</u> Rouille, Lyon 1556 (`btv1b22000559`, API BnF)

In [ ]:
telecharger_api_bnf("btv1b22000559", "bois_eskrich_rouille_lyon1556", prefixe="eskrich")

### Leroy II, Guillaume
##### <u>Édition :</u> Gueynard, Lyon 1510 (`bsb11054210`, BSB Munich)

In [ ]:
telecharger_bsb("bsb11054210", "bois_leroy_gueynard_lyon1510", prefixe="leroy")

### Biblioteca Digital Ovidiana (ovidiuspictus.es)

Source complémentaire utilisée pour récupérer des illustrations non accessibles
via l'API BnF ou BSB Munich.

La **Biblioteca Digital Ovidiana** est un projet de recherche espagnol qui recense
et numérise les éditions illustrées des *Métamorphoses* d'Ovide du 15e au 18e siècle.
Chaque édition indexée dispose d'une page dédiée listant ses illustrations sous forme
de vignettes téléchargeables.

**Avantage :** les illustrations sont déjà découpées et présentées individuellement —
pas besoin de segmentation YOLO.

**Limite :** certaines pages incluent des frontispices ou illustrations de titre
qui ne correspondent pas à des scènes des fables ovidiennes — à exclure manuellement.

**Accès :** `http://www.ovidiuspictus.es` — images téléchargeables via web scraping
depuis les pages `ilustracionesejemplar.php?clave={id}`.

### Monogrammiste H.T.
##### <u>Édition :</u> Molin, Lyon 1697 — tomes 4/5/6 (claves 52/53/54, Biblioteca Digital Ovidiana)

In [ ]:
telecharger_depuis_clave(52, "cuivre_ht_molin_lyon1697")
telecharger_depuis_clave(53, "cuivre_ht_molin_lyon1697")
telecharger_depuis_clave(54, "cuivre_ht_molin_lyon1697")

### Franco
##### <u>Édition :</u> Giunta, Venise 1584 (clave=21, Biblioteca Digital Ovidiana)

In [ ]:
telecharger_depuis_clave(21, "cuivre_franco_giunta_venise1584")

### Gaultier, Léonard
##### <u>Éditions :</u> Guillemot, Paris 1610 (`bsb11913284`, BSB Munich) · Veuve Guillemot, Paris 1614 (PDF local)

In [ ]:
telecharger_bsb("bsb11913284", "cuivre_gaultier_guillemot_paris1610", prefixe="gaultier")
traiter_pdf_local("cuivre_gaultier_veuveguillemot_paris1614.pdf", "cuivre_gaultier_veuveguillemot_paris1614")

### Isaac, Jaspar
##### <u>Édition :</u> L'Angelier, Paris 1617 (PDF local, renommé depuis Gallica — voir « Préparation des PDFs locaux »)

In [ ]:
traiter_pdf_local("cuivre_isaac_langelier_paris1617.pdf", "cuivre_isaac_langelier_paris1617")

### Briot, Isaac
##### <u>Édition :</u> Drobet, Lyon 1628 (PDF local)

In [ ]:
traiter_pdf_local("briot_drobet_lyon1628.pdf", "cuivre_briot_drobet_lyon1628")

### Baur, Johann Wilhelm
##### <u>Éditions :</u> s.n., Augsbourg 1709 (`bsb10872075`, BSB Munich) · s.n., Vienne 1639 (`bsb10872073`, BSB Munich)

In [ ]:
telecharger_bsb("bsb10872075", "cuivre_baur_sn_augsbourg1709", prefixe="baur")
telecharger_bsb("bsb10872073", "cuivre_baur_sn_vienne1639", prefixe="baur")

### Altzenbach, Gerhardt
##### <u>Édition :</u> Cöllen 1681 — exemplaire physique Bnu Strasbourg, à numériser. Pas encore de version numérique disponible.

### Goltzius, Hendrick
##### <u>Édition :</u> Haarlem 1589 (PDF local) — planches en couleur, converties en niveaux de gris pour homogénéité avec le reste du corpus.

In [ ]:
traiter_pdf_local("goltzius_haarlem1589.pdf", "cuivre_goltzius_goltzius_haarlem1589")

In [ ]:
convertir_niveaux_de_gris("cuivre_goltzius_goltzius_haarlem1589")

### Mulder, Joseph
##### <u>Édition :</u> Amsterdam 1683 — à sourcer, aucune édition numérisée trouvée à ce jour (ressemble à Philippe).

## 5. Libérer la mémoire GPU


In [ ]:
modele_yolo = liberer_yolo(modele_yolo)

## 6. Récapitulatif des illustrations disponibles

In [5]:
print("Illustrations disponibles par graveur :\n")

grand_total = 0
for graveur, dossiers in GRAVEURS.items():
    print(f"━━ {graveur.upper()} ━━")
    total = 0
    for d in dossiers:
        chemin = os.path.join(DOSSIER_SEG, d)
        if os.path.exists(chemin):
            n = len([f for f in os.listdir(chemin)
                     if f.endswith(".jpg") and "_flip" not in f])
            total += n
            print(f"    {d:50s} : {n:4d}")
        else:
            print(f"    ⚠️  {d:50s} : introuvable")
    print(f"    {'→ TOTAL ' + graveur:50s} : {total:4d}\n")
    grand_total += total

print("=" * 62)
print(f"  {len(GRAVEURS)} graveurs · {grand_total} illustrations au total")

Illustrations disponibles par graveur :

━━ SALOMON ━━
    bois_salomon_rouille_lyon1557                      :  161
    → TOTAL salomon                                    :  161

━━ SOLIS ━━
    bois_solis_feyerabend_francfort1581                :  184
    → TOTAL solis                                      :  184

━━ WICKRAM ━━
    bois_wickram_behem_mayence1545                     :   50
    → TOTAL wickram                                    :   50

━━ SAVERY ━━
    cuivre_savery_farnaby_paris1637                    :   18
    → TOTAL savery                                     :   18

━━ DE_PASSE ━━
    cuivre_depasse_depasse_koln1602                    :  134
    cuivre_depasse_jansonius_arnhem1607                :  136
    → TOTAL de_passe                                   :  270

━━ ESKRICH ━━
    bois_eskrich_rouille_lyon1556                      :   42
    → TOTAL eskrich                                    :   42

━━ LEROY ━━
    bois_leroy_gueynard_lyon1510                     